# Article figures — simulation panels

**This notebook does no analysis.** It loads pre-computed tables and renders one panel per file.
It merges the former main-text, SI and revision figure notebooks (2026-10-28). For the
revision, the original simulation figure is **split by system,
not by method**: the apo protein, the 1:1 Nhp6A:DNA complex (with the IDR-only control), and the
2:1 complex each get their own figure; the mutant and $Q_\mathrm{BH}$ panels are SI.

**Every code cell is self-contained.** Each cell carries its own imports, style, limits, colours and
output name, so you can restart the kernel, run one cell, and get one figure. Each cell starts by
locating the project root (the first ancestor holding `analysis/scripts/`) and `chdir`-ing there,
so all paths below are project-root relative and the notebook runs from any working directory or
host. The only shared code is `analysis/scripts/article_plot.py` (loading, reliability truncation,
file naming; no styling).

**Output.** Every panel is written as **PDF + PNG** (300 dpi) to `figures/article/`, named
`<system>_<observable>[_<variant>]_v<N>` with no figure number, so names survive renumbering.
`VERSION` is carried over from the pre-merge notebooks: it tracks the data behind a panel, not the
figure it lands in. The pre-merge renders are kept in `figures/article/premerge/`.

| Section | Panel | Table | Output (`figures/article/`) |
|---|---|---|---|
| Apo Nhp6A | A | `analysis/article/A_fes2d_qcore_nidr.dat` | `1Nhp6A_apo_large_fes2d_qcore_nidr` |
| | B | `analysis/article/B_helicity_perhelix_vs_qcore.dat` | `1Nhp6A_apo_large_helicity_perhelix_vs_qcore` |
| | C | `analysis/article/C_pmf2d_qcore_total.dat` + `C_medoids_cvs.dat` | `1Nhp6A_apo_large_pmf2d_qcore_helicity` |
| 1:1 Nhp6A:DNA | D1 (rev. + IDR) | `analysis/article/D_pmf_bend_{DNA,holo,idr}.dat` | `1Nhp6A_wall_pmf_bend_vs_dna_idr` |
| | D2 | `analysis/article/D_pmf_de2e_{DNA,holo}.dat` | `1Nhp6A_wall_pmf_de2e_vs_dna` |
| | E | `analysis/article/E_pmf2d_idrdna_bend.dat` | `1Nhp6A_wall_pmf2d_idrdna_bend` |
| | E-IDR (rev.) | `analysis/1IDR_wall_pmf2d_idrdna_bend.dat` + `analysis/snapshots/1IDR_wall_bend_medoids_cvs.dat` | `1IDR_wall_pmf2d_idrdna_bend` |
| 2:1 complex | F1 | `analysis/article/F1_equil_2Nhp6A_h3.dat` | `2Nhp6A_equil_h3_distortion` |
| | F1 rep2 | `analysis/article/F1_equil_2Nhp6A_h3_rep2.dat` | `2Nhp6A_equil_h3_distortion_rep2` |
| | F2 | `analysis/article/F2_equil_bend_hist.dat` | `2Nhp6A_equil_bend_compare` |
| SI — mutants | S1 | `analysis/<sys>_pmf2d_idrdna_bend.dat` | `mutants_pmf2d_idrdna_bend` |
| | S2 | `analysis/<sys>_pmf_bend_mean_std.dat` | `mutants_pmf_bend_vs_wt_dna` |
| | S3 | `analysis/<sys>_pmf_nidr_mean_std.dat` | `mutants_pmf_nidr_vs_wt` |
| SI — $Q_\mathrm{BH}$ | A/B/C on $Q_\mathrm{BH}$ | `analysis/article/{A,B,C}_*_qbh*.dat` | `1Nhp6A_apo_large_*_qbh_*` |
| — | provenance stamp | `analysis/article/meta.json` | `article_figures_provenance.txt` |

The tables come from (re-run these when simulation data change; re-run *this* notebook to change
how a panel looks):

```
python analysis/scripts/article_data.py                                   # A-F tables (~20 min)
python analysis/scripts/article_data.py --panels D --d-systems idr        # IDR curve only
python analysis/scripts/dna_bending_wall_compare.py --idrdna-bend-2d 1IDR_wall
python analysis/scripts/dna_bending_wall_compare.py                       # mutant SI tables (run_all)
python select_snapshots.py --system 1IDR_wall_bend --barrier 1.0 --f-ceil 8.0 --min-cells 6 --t-max-ns 300 --n-per-state 200 --medoid-cv-tiebreak
```

**Conventions applied at plot time** (the `.dat` files stay full-range, by design): a sampling mask
(`MIN_VISITS`), the reliability truncation (std ≤ `STD_CEIL`, contiguous from the minimum), a
per-bin ESS floor for reweighted means, and — in the SI and E-IDR panels — the walker-ergodicity
mask (`MIN_WALKERS`). Every curve carries a mean ± std band. The N-terminal tail is labelled **IDR**.

### Provenance of the data being plotted

Optional — run it to see which dataset, how long, and which reweighting gates each panel rests on.

In [ ]:
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import article_plot as f5

META = f5.load_meta()
print(f"data generated {META['generated']} on {META['host']} "
      f"| stride {META['stride']} | {META['n_snaps']} snapshots")
print(f"DNA-only source: {META['dna_source']}")
print("panels last recomputed:", ", ".join(f"{k} {v}" for k, v in
                                           sorted(META.get("panels_last_run", {}).items())))
if "reweighting_apo" in META:
    r = META["reweighting_apo"]
    print(f"\napo c(t) reweighting: ESS = {r['ess']:,.0f} ({r['ess_percent']}%)  "
          f"gate MAD bulk = {r['gate_mad_qcore_bulk']} kJ/mol  -> "
          f"{'PASS' if r['gate_pass'] else 'FAIL'}")
for k, v in META.get("D", {}).items():
    print(f"{v['label']:12s} {v['ns_per_walker']:.0f} ns/walker  ESS {v['ess_percent']}%  "
          f"MAD(de2e) {v['gate_mad_de2e']} kJ/mol  -> {'OK' if v['gate_pass'] else 'CHECK'}")
print()
for f in META["flags"]:
    print("FLAG:", f)

## Apo Nhp6A — folded-core stability and helicity

Protein alone, `1Nhp6A_apo_large` well-tempered MetaD, 1250 ns/walker.

### Panel A — apo $F(Q_\mathrm{core},\, n_\mathrm{IDR-core})$

*"In the apo form, the native state ensemble predominantly involves fluctuations in the number of
IDR–folded core contacts … unstructuring (decrease in Q) is possible."*

Both CVs are biased, so this is `sum_hills`, no reweighting involved.

In [ ]:
# ─── Panel A: everything this figure needs ────────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — A
VERSION    = "v4"                # v4: apo COMPLETE at 7.5 us (1250 ns/walker); panel tables rebuilt 2026-08-06
FIGSIZE    = (4.2, 3.6)          # inches (single-column panel + colourbar;
                                 #  tall enough that the square axes box is not tiny)
XLIM       = (0.80, 1.00)        # Q_core (matched to panel C)
YLIM       = (0, 50)             # n_IDR-core
VMAX       = 50                  # kJ/mol colour ceiling
CMAP       = "RdYlBu_r"
STD_CEIL   = 25.0                # kJ/mol — truncate above this cross-snapshot std
MIN_VISITS = 5                   # colvar frames per bin below which the bin is left white
SMOOTH_SIGMA = 0.8               # display-only Gaussian smoothing of the contours, in bins
                                 # (0 = off; mask-preserving, does not move the minimum)
# NOTE: state medoids are deliberately NOT marked on this panel. They are segmented on the
# S_alpha-Q_core surface (panel C) and their n_IDR-core coordinate is single-walker-dominated
# (folded-basin structural pool is ~13/15 from replica 0) and NOT converged (MAD(n_tail_core)
# = 2.6 kJ/mol). Marking them here would plant the "folded" label in a high-F, non-converged
# tail of this surface. Medoid markers live on panel C only, where they were defined.
TITLE      = ""                  # "" for the assembled figure
OUT        = "1Nhp6A_apo_large_fes2d_qcore_nidr"
plt.rcParams.update({"font.size": 15, "axes.labelsize": 12, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "axes.linewidth": 0.8, "figure.dpi": 110,
                     "axes.grid": False, "pdf.fonttype": 42})

q, ntc, mean, std, cnt = f5.load2d("A_fes2d_qcore_nidr.dat")
F = f5.reliable_2d(mean, std, cnt, STD_CEIL, MIN_VISITS)
Fp = f5.smooth_masked(F, SMOOTH_SIGMA)      # smoothed only for drawing; F stays for the numbers

fig, ax = plt.subplots(figsize=FIGSIZE)
Fm = np.ma.masked_invalid(Fp)
im = ax.contourf(q, ntc, Fm.T, levels=np.linspace(0, VMAX, 21), cmap=CMAP, extend="max")
ax.contour(q, ntc, Fm.T, levels=np.arange(5, VMAX, 5), colors="k", linewidths=0.35, alpha=0.45)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$\Delta F$ (kJ mol$^{-1}$)")
cb.ax.tick_params(labelsize=8)
ax.set_box_aspect(1)            # square AXES box; the colourbar takes its space from the
                                # figure, not from the box (set_aspect would mix CV units)
ax.set_xlabel(r"$Q_\mathrm{core}$")
ax.set_ylabel(r"$n_\mathrm{IDR-core}$")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

i, j = np.unravel_index(np.nanargmin(F), F.shape)
print(f"  minimum at Q_core = {q[i]:.3f}, n_IDR-core = {ntc[j]:.0f}")
print(f"  largest std shown: {np.nanmax(np.where(np.isfinite(F), std, np.nan)):.1f} kJ/mol")

### Panel B — per-helix helicity vs $Q_\mathrm{core}$

*"helicity is reduced mostly in helix 2, which is involved in the binding to DNA in the major mode."*

The three ALPHARMSD values have different maxima (they scale with helix length), so each is
normalised by its **folded-state** value — that is what makes "reduced *mostly* in H2" a like-for-like
comparison. Set `NORM_MODE = "none"` for raw $S_\alpha$.

In [ ]:
# ─── Panel B: everything this figure needs ────────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — B
VERSION     = "v4"                # v4: apo COMPLETE at 7.5 us (1250 ns/walker); panel tables rebuilt 2026-08-06
FIGSIZE     = (3.3, 2.)
XLIM        = (0.80, 1.00)      # Q_core (matched to panels A and C)
NORM_MODE   = "folded"          # "folded" (relative to the folded state) | "max" | "none"
ESS_MIN_BIN = 30                # per-bin effective sample size floor for a reweighted mean
COLOR       = {"helix1": "#4C72B0", "helix2": "#C44E52", "helix3": "#55A868"}
#LABEL       = {"helix1": "H1 (27–42)", "helix2": "H2 (48–61)", "helix3": "H3 (64–91)"}
LABEL       = {"helix1": "H1", "helix2": "H2", "helix3": "H3"}
S_MAX       = {"helix1": 12.0, "helix2": 10.0, "helix3": 24.0}   # ~ n_res - 5, for NORM_MODE="max"
TITLE       = ""
OUT         = "1Nhp6A_apo_large_helicity_perhelix_vs_qcore"
plt.rcParams.update({"font.size": 12, "axes.labelsize": 12, "xtick.labelsize": 10,
                     "ytick.labelsize": 10, "legend.fontsize": 10, "axes.linewidth": 0.8,
                     "figure.dpi": 150, "axes.grid": False, "pdf.fonttype": 42})

b = f5.load1d("B_helicity_perhelix_vs_qcore.dat")
with open("analysis/article/B_helicity_perhelix_vs_qcore.dat") as fh:
    ref_line = [l for l in fh if "folded-state reference" in l][0]
ref = {k: float(v) for k, v in (t.split("=") for t in ref_line.split(":")[1].split())}
print("folded-state <S_alpha>:", "  ".join(f"{k}={v:.2f}" for k, v in ref.items()))

fig, ax = plt.subplots(figsize=FIGSIZE)
for h in ["helix1", "helix2", "helix3"]:
    scale = {"folded": ref[h], "max": S_MAX[h], "none": 1.0}[NORM_MODE]
    m, sd = b[f"{h}_mean"] / scale, b[f"{h}_std"] / scale
    ok = (np.isfinite(m) & (b["ess_bin"] >= ESS_MIN_BIN)
          & (b["q_core"] >= XLIM[0]) & (b["q_core"] <= XLIM[1]))
    ax.plot(b["q_core"][ok], m[ok], "-o", ms=3, lw=1.6, color=COLOR[h], label=LABEL[h])
    ax.fill_between(b["q_core"][ok], (m - sd)[ok], (m + sd)[ok], color=COLOR[h], alpha=0.22, lw=0)
    print(f"  {LABEL[h]:12s} Q={b['q_core'][ok][0]:.2f}: {100 * m[ok][0]:5.1f}%   "
          f"Q={b['q_core'][ok][-1]:.2f}: {100 * m[ok][-1]:5.1f}%  (of folded)")

ax.set_xlabel(r"$Q_\mathrm{core}$")
ax.set_ylabel({"folded": r"$\langle S_\alpha\rangle\,/\,\langle S_\alpha\rangle_\mathrm{folded}$",
               "max": "fractional helicity",
               "none": r"$\langle S_\alpha\rangle$"}[NORM_MODE])
ax.set_xlim(*XLIM)
if NORM_MODE == "folded":
    ax.axhline(1.0, color="0.6", lw=0.7, ls=":", zorder=0)
ax.legend(frameon=False, loc="lower right")
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE, transparent=False)   # opaque white background (user request)
plt.show()

### Panel C — apo $F(Q_\mathrm{core},\, S_\alpha^{\mathrm{total}})$: one basin, plus a shoulder

The 2D version of the panel-B claim, on the **total** helicity of the HMG box
($S_\alpha^{\mathrm{tot}} = S_\alpha^{\mathrm{H1}} + S_\alpha^{\mathrm{H2}} + S_\alpha^{\mathrm{H3}}$),
so the axis is the overall helical content rather than one helix's.

### There is only one basin

At 5.06 µs (867 ns/walker) the surface has **a single free-energy basin**, at
$Q_\mathrm{core} \approx 1$, $S_\alpha^{\mathrm{tot}} \approx 40$. The "second state" of the earlier
version does not survive the noise floor and has been **demoted to a shoulder** — it is *not* a
metastable state, and must not be described as one.

The evidence, all on this same reweighted surface:

| | $\Delta F$ (kJ/mol) | std | counts |
|---|---|---|---|
| basin minimum | 0.00 | 0.26 | 803,705 |
| shoulder minimum | 13.61 | 0.42 | 120,433 |
| the surface *between* them | 12.58 | 3.30 | 34,885 |

The point separating them is **lower** than the shoulder itself, so nothing encloses the shoulder.
A prominence scan makes it quantitative: the raw watershed seeds 5 minima, but the count collapses
to **1 as soon as the required saddle prominence reaches 2 kJ/mol, and stays at 1 up to 10 kJ/mol**.
Every candidate second basin is separated by **less than $k_BT$ (2.49 kJ/mol)** — i.e. by less than
this surface's own $\pm 4$–$6$ kJ/mol cross-snapshot uncertainty. This is not a sampling problem:
the shoulder is *well* sampled (120k frames, std 0.42 kJ/mol). The old two-state reading was an
artefact of segmenting with no prominence threshold, which is also why its medoid kept landing off
its "well" — there was no well.

Nor was it lost to the reliability mask: at $Q_\mathrm{core} = 0.87$, $S_\alpha = 36.5$ it sits well
inside the reliable region.

### What the shoulder is

The shoulder minimum lies **13.61 kJ/mol = 5.46 $k_BT$** above the global minimum and carries
**≈ 0.5 %** of the reweighted population of the reliable region (integrated over the window below;
the Boltzmann factor at its lowest bin alone is 0.43 %).

Structurally it is **H2 melting**, and only H2 — comparing the two medoids, H1 (8.70 → 9.47) and
H3 (20.99 → 20.25) are unchanged, while **H2 collapses from 8.70 to 5.15**.

The markers are therefore *not* two basin medoids. They are:

* **1** (circle) — medoid of the one basin: `analysis/article/state_1_medoid.pdb`
* **2** (diamond) — representative of the **shoulder region**: `analysis/article/state_2_medoid.pdb`

Marker 2 is a diamond precisely so the panel does not imply a second basin. The shoulder is defined
as an explicit CV window (centre $S_\alpha = 36.5$, $Q_\mathrm{core} = 0.865$; half-widths 2.0 and
0.02) rather than as a detected basin, because a prominence threshold above the noise floor
correctly merges it away. Both are structural medoids (lowest mean heavy-atom RMSD to the rest of
their region). Regenerate with

```
python select_snapshots.py --system 1Nhp6A_apo_large_helicity \
    --f-ceil 25 --barrier 5 --delta-f 3 --t-max-ns 900 --shoulder 36.5,0.865,2.0,0.02
```

`--barrier 5` is what keeps the watershed from picking a valley-floor ripple as a "state"; `--shoulder`
then re-adds the shoulder as an explicitly labelled region, reported as `SHOULDER (not a basin)`.

### Axes and the $Q_\mathrm{core} = 1$ edge

The panel is drawn on panel A's $Q_\mathrm{core}$ window (widened slightly so neither marker is
clipped). Note this crops the surface: the reliable region actually extends down to
$Q_\mathrm{core} \approx 0.59$. Set `XLIM = (0.58, 1.01)` to see that whole melting branch.

The grid is carried out to the bins' outer **edges** (`f5.to_bin_edges`) before contouring, so the
surface reaches the physical bound $Q_\mathrm{core} = 1$ instead of stopping at the last bin centre
(0.99) — that is where the folded state, and its medoid at $Q_\mathrm{core} = 0.9995$, actually sit.
No value is invented: each border bin is extended outward by half its width.

**Why the folded medoid sits at $Q_\mathrm{core} \approx 1$ and not a little below.** `q_core` is a
Best-Hummer contact map with a *soft* switch ($\beta = 50$ nm$^{-1}$, $\lambda = 1.4$): a native
contact with reference distance 0.619 nm is only half-broken at $\lambda \times \mathrm{REF} =
0.867$ nm. In the folded medoid the largest of the 67 native C$\alpha$–C$\alpha$ distances is
0.799 nm, so every contact still scores $s \geq 0.996$ and $Q = 0.9998$. Thermal fluctuations of the
folded core simply do not move this CV — it saturates, and the folded ensemble's median is 0.9987.
$Q_\mathrm{core}$ only responds once contacts genuinely break. This is *not* the same quantity as the
`cmap` CV of the bound systems, which uses a much sharper 0.30 nm contact definition and therefore
sits at ~0.96 when folded.

Checked, not assumed: recomputing $Q$ from the medoid PDBs with the switching parameters read
straight out of `plumed_meta_2D_1Nhp6A_large.dat` reproduces the colvar values to four decimals,
which also confirms the frame-to-structure mapping in `select_snapshots.py`.

To show one helix instead, re-run `article_data.py --panels C --helix helix2` and point `CFILE` at
`C_pmf2d_qcore_helix2.dat` (the markers are defined on total helicity, so switch them off).

In [ ]:
# ─── Panel C: everything this figure needs ────────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — C
VERSION    = "v6"                        # v6: pool extended 900 -> 1250 ns (repointed to analysis/pbc/solute_r*.xtc; the old _proc.xtc stopped at 1000 ns) AND the medoid restricted to the folded core, res 21-93 -- on all 761 heavy atoms the disordered IDR decided the medoid outright and put marker 1 at q_core 0.9674. n_per_state 15 -> 200 so the top of the RMSD ranking is homogeneous, then --medoid-cv-tiebreak --medoid-tol-pct 10 keeps, among those structurally equivalent frames, the one nearest each region's free-energy minimum. Surface unchanged (1250 ns).
CFILE      = "C_pmf2d_qcore_total.dat"   # total helicity = H1+H2+H3
FIGSIZE    = (4.2, 3.6)
# Panel A's Q window, widened at both ends so neither marker is clipped: the shoulder sits at
# Q=0.859, the basin medoid at Q=0.9998. The reliable region actually runs down to Q=0.59 --
# set XLIM=(0.58, 1.01) to see the whole melting branch.
XLIM       = (0.80, 1.00)     # Q_core — ends on the physical bound Q=1, where the surface ends
YLIM       = None             # None -> fit the reliable region inside XLIM; or e.g. (26, 44)
VMAX       = 50               # kJ/mol colour ceiling
CMAP       = "RdYlBu_r"
STD_CEIL   = 15.0
MIN_VISITS = 5
# The smoothing now lives in the ANALYSIS: article_data.py builds this surface on a fine grid
# (140x92: dq=0.005, dS_alpha=0.5) and Gaussian-smooths the reweighted DENSITY before the log
# (KDE bandwidth (0.01, 0.8) in CV units). Smoothing the density, not F, keeps barriers unbiased.
# So no display-time blur is needed here -- 0 avoids double-smoothing. (Set >0 only if you
# regenerate with --c-bw 0 0, i.e. a raw fine histogram.)
SMOOTH_SIGMA = 0.0
SHOW_MEDOIDS = True
# Marker 1 = the medoid of the ONE basin. Marker 2 = a representative of the SHOULDER, which is
# not a basin (its saddle prominence is < kBT; see the markdown above). They are drawn with
# the same white circle as the basin (user 2026-11-04); the caption/markdown must say it is a shoulder.
SHOULDER_ID   = 2
# Numbers drawn INSIDE the markers, same style as panels E-IDR / E-Δ2-10.
BASIN_STYLE    = dict(marker="o", ms=13, mfc="white", mec="black", mew=1.0, ls="none", zorder=5,
                      clip_on=False)   # the basin medoid sits at Q=0.9998, right on the axis spine
SHOULDER_STYLE = dict(marker="o", ms=13, mfc="white", mec="black", mew=1.0, ls="none", zorder=5,
                      clip_on=False)
TITLE      = ""
OUT        = "1Nhp6A_apo_large_pmf2d_qcore_helicity"
#plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
#                     "ytick.labelsize": 9, "axes.linewidth": 0.8, "figure.dpi": 110,
#                     "axes.grid": False, "pdf.fonttype": 42})
plt.rcParams.update({"font.size": 15, "axes.labelsize": 12, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "axes.linewidth": 0.8, "figure.dpi": 110,
                     "axes.grid": False, "pdf.fonttype": 42})
q, sa, mean, std, cnt = f5.load2d(CFILE)
F = f5.reliable_2d(mean, std, cnt, STD_CEIL, MIN_VISITS)
Fp = f5.smooth_masked(F, SMOOTH_SIGMA)   # analysis already KDE-smoothed; SMOOTH_SIGMA=0 -> no-op
# contourf paints on the bin CENTRES, so the map would stop at Q_core = 0.99 -- half a bin short of
# the Q = 1 bound, where the folded state actually lives. Carry the grid out to the bin edges.
qe, sae, Fe = f5.to_bin_edges(q, sa, Fp)

fig, ax = plt.subplots(figsize=FIGSIZE)
Fm = np.ma.masked_invalid(Fe)
im = ax.contourf(qe, sae, Fm.T, levels=np.linspace(0, VMAX, 21), cmap=CMAP, extend="max")
ax.contour(qe, sae, Fm.T, levels=np.arange(5, VMAX, 5), colors="k", linewidths=0.35, alpha=0.45)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$\Delta F$ (kJ mol$^{-1}$)")
cb.ax.tick_params(labelsize=8)
ax.set_box_aspect(1)            # square AXES box; the colourbar takes its space from the
                                # figure, not from the box (set_aspect would mix CV units)
ax.set_xlabel(r"$Q_\mathrm{core}$")
ax.set_ylabel(r"$S_\alpha^\mathrm{total}$  (H1+H2+H3)")
ax.set_xlim(*XLIM)
if YLIM:
    ax.set_ylim(*YLIM)
else:                     # fit the reliable region *within* XLIM
    rows = np.isfinite(Fe[(qe >= XLIM[0]) & (qe <= XLIM[1])]).any(0)
    ax.set_ylim(sae[rows].min(), sae[rows].max())

# the structures shown as snapshots, at their true (Q_core, S_alpha) coordinates
if SHOW_MEDOIDS:
    state, hel, qc, names = f5.load_medoids("C_medoids_cvs.dat")   # cv1=total_helicity, cv2=q_core
    for s, h, x in zip(state, hel, qc):
        is_sh = int(s) == SHOULDER_ID
        ax.plot([x], [h], **(SHOULDER_STYLE if is_sh else BASIN_STYLE))
        ax.text(x, h, str(s), ha="center", va="center", fontsize=9, fontweight="bold",
                color="black", zorder=6, clip_on=False)
        kind = "SHOULDER (not a basin)" if is_sh else "basin medoid"
        print(f"  {s}: Q_core = {x:.4f}, S_alpha_total = {h:.1f}  [{kind}]"
              f"  -> analysis/article/state_{s}_medoid.pdb")
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

i, j = np.unravel_index(np.nanargmin(F), F.shape)
print(f"  minimum at Q_core = {q[i]:.3f}, S_alpha_total = {sa[j]:.1f}")

## 1:1 Nhp6A:DNA — DNA bending and the IDR (+ IDR-only and Δ2-10 revision panels)

Free DNA (`DNA_wall2`), the 1:1 complex (`1Nhp6A_wall`) and the IDR-only peptide on the same DNA
(`1IDR_wall`, Nhp6A residues 1–23, C-amidated). Each system uses its full dataset (no
time-matching): 1IDR 400 ns/walker (complete run, recomputed 2026-11-03); the holo D/E tables are
still 300 ns/walker (the holo run has since grown to 350 ns, but D1/E were not recomputed, so the
published WT curves do not move). $n_\mathrm{IDR-DNA}$ is the
identical CV in both systems (Cα+Cβ of residues 1–20 vs the DNA P/C1′ atoms). The bend angle is an
unbiased spectator: every bend surface is final-bias reweighted, never `sum_hills`.

The original D1 (v2, DNA vs holo only) is superseded by the revised D1 below, whose DNA and holo
curves are identical; set its `SYS` to the first two entries to reproduce v2.

### Reweighting gate of the D-panel systems (incl. IDR-only)

In [ ]:
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import article_plot as f5

META = f5.load_meta()
for k in ("DNA", "holo", "idr"):
    d = META["D"].get(k)
    if d is None:
        print(f"{k:5s} MISSING -> python analysis/scripts/article_data.py --panels D --d-systems {k}")
        continue
    print(f"{k:5s} {d['label']:11s} {d['data']:18s} {d['ns_per_walker']:6.0f} ns/walker  "
          f"ESS {d['ess_percent']:5.1f}%  MAD(de2e) {d['gate_mad_de2e']:.2f} kJ/mol  "
          f"-> {'PASS' if d['gate_pass'] else 'FAIL'}")

### Panel D1 (revised) — DNA bend angle: free DNA, Nhp6A-bound, IDR-bound

Answers the request for the IDR-only bending PMF. Identical to Panel D1 v2
(same tables, axes, colours, truncation) with the IDR-only curve added. The IDR alone does
**not** bend the DNA: its minimum stays at ~8° and its profile sits on the free-DNA one up to ~30°,
whereas full-length Nhp6A shifts the minimum to ~30°. At 400 ns the IDR profile is ~5 kJ/mol below
free DNA at 40° (23.8 ± 2.2 vs 29.3 ± 1.8 kJ/mol; the gap grew from 300 ns, mostly from walkers r0/r4),
so the IDR softens strong bends somewhat but creates no bent minimum. Read together with panel E: IDR–DNA contacts are necessary for the bent state
of the complex but not sufficient — the docked HMG box is required (cf. review item A10: the IDR
*modulates* the bend, it does not create it; consistent with Yen 1998 / Masse 2002).

In [ ]:
# ─── Panel D1 (revised): everything this figure needs ─────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — D1 (revised)
VERSION  = "v4"          # v4: IDR curve at 400 ns/walker (full run); v3: IDR at 300 ns; DNA/holo curves identical to v2
FIGSIZE  = (3.3, 2.7)
XLIM     = (0, 60)          # degrees
YLIM     = (0, 20)          # kJ/mol
STD_CEIL = 15.0             # kJ/mol — reliability truncation (as Panel D1)
SYS      = {"DNA":  dict(label="DNA alone", color="#0072B2", ls="-"),      # Okabe-Ito
            "holo": dict(label="Nhp6A:DNA", color="#D55E00", ls="-"),
            "idr":  dict(label="IDR:DNA",   color="#000000", ls="--")}   # 1IDR colour of dna_bending_wall_compare; green is 2Nhp6A in F2
TITLE    = ""
OUT      = "1Nhp6A_wall_pmf_bend_vs_dna_idr"
plt.rcParams.update({"font.size": 12, "axes.labelsize": 10, "xtick.labelsize": 10,
                     "ytick.labelsize": 10, "legend.fontsize": 10, "axes.linewidth": 0.8,
                     "figure.dpi": 150, "axes.grid": False, "pdf.fonttype": 42})

fig, ax = plt.subplots(figsize=FIGSIZE)
for name, s in SYS.items():
    d = f5.load1d(f"D_pmf_bend_{name}.dat")
    x, m, sd = d[list(d)[0]], d["F_mean"], d["F_std"]
    ok = f5.reliable_1d(m, sd, STD_CEIL)
    ax.plot(x[ok], m[ok], lw=2, color=s["color"], ls=s["ls"], label=s["label"])
    ax.fill_between(x[ok], (m - sd)[ok], (m + sd)[ok], color=s["color"], alpha=0.25, lw=0)
    print(f"  {s['label']:11s} minimum at {x[ok][np.nanargmin(m[ok])]:5.1f} deg"
          f"   (reliable {x[ok].min():.1f} – {x[ok].max():.1f})")

ax.set_xlabel("DNA bend angle (deg)")
ax.set_ylabel(r"$\Delta F$ (kJ mol$^{-1}$)")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.legend(frameon=False, loc="upper left")
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

### Panel D1-Δ2-10 — DNA bend angle: free DNA, Nhp6A-bound, IDR-bound, Δ2-10-bound

Panel D1 (revised) with the Δ2-10 deletion added: same tables, axes, truncation and colours for the
first three curves (they are byte-identical to D1 v4), plus `D_pmf_bend_d2_10.dat` from
`article_data.py --panels D --d-systems d2_10` (full run, 350 ns/walker; final-bias reweighted,
5-snapshot mean ± std, exactly as the other three).

**Δ2-10 fails the reweighting gate** (ESS 19 %, MAD(de2e) 4.30 kJ/mol > $k_BT$), so its curve is
drawn dash-dot and must be read as indicative, not quoted at the precision of the others. Its
minimum sits at ~28°, i.e. with full-length Nhp6A (~30°), not with free DNA / IDR (~8°): deleting
residues 2–10 does not abolish bending.


In [ ]:
# ─── Panel D1-Δ2-10: everything this figure needs ─────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — D1-Δ2-10 (matched to D1 revised v4)
VERSION  = "v1"          # v1: D1 v4 + Δ2-10 at 350 ns/walker (full run; reweighting gate FAILS)
FIGSIZE  = (3.3, 2.7)
XLIM     = (0, 60)          # degrees
YLIM     = (0, 20)          # kJ/mol
STD_CEIL = 15.0             # kJ/mol — reliability truncation (as Panel D1)
SYS      = {"DNA":  dict(label="DNA alone", color="#0072B2", ls="-"),      # Okabe-Ito
            "holo": dict(label="Nhp6A:DNA", color="#D55E00", ls="-"),
            "idr":  dict(label="IDR:DNA",   color="#000000", ls="--"),   # 1IDR colour of dna_bending_wall_compare; green is 2Nhp6A in F2
            "d2_10": dict(label="Δ2-10:DNA", color="#882255", ls="-.")}  # 1D2_10 colour of dna_bending_wall_compare; dash-dot = gate fails
TITLE    = ""
OUT      = "1Nhp6A_wall_pmf_bend_vs_dna_idr_d2_10"
plt.rcParams.update({"font.size": 12, "axes.labelsize": 10, "xtick.labelsize": 10,
                     "ytick.labelsize": 10, "legend.fontsize": 10, "axes.linewidth": 0.8,
                     "figure.dpi": 150, "axes.grid": False, "pdf.fonttype": 42})

fig, ax = plt.subplots(figsize=FIGSIZE)
for name, s in SYS.items():
    d = f5.load1d(f"D_pmf_bend_{name}.dat")
    x, m, sd = d[list(d)[0]], d["F_mean"], d["F_std"]
    ok = f5.reliable_1d(m, sd, STD_CEIL)
    ax.plot(x[ok], m[ok], lw=2, color=s["color"], ls=s["ls"], label=s["label"])
    ax.fill_between(x[ok], (m - sd)[ok], (m + sd)[ok], color=s["color"], alpha=0.25, lw=0)
    print(f"  {s['label']:11s} minimum at {x[ok][np.nanargmin(m[ok])]:5.1f} deg"
          f"   (reliable {x[ok].min():.1f} – {x[ok].max():.1f})")

ax.set_xlabel("DNA bend angle (deg)")
ax.set_ylabel(r"$\Delta F$ (kJ mol$^{-1}$)")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.legend(frameon=False, loc="upper left")
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

In [ ]:
# ─── Panel D2-Δ2-10: F(n_IDR-DNA), companion to Panel D1-Δ2-10 ─────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — D2-Δ2-10 (matched to D1-Δ2-10 v1; no "DNA alone" curve: n_IDR-DNA undefined)
VERSION  = "v1"          # v1: holo 350, IDR 400, Δ2-10 350 ns/walker (full runs; Δ2-10 gate FAILS)
FIGSIZE  = (3.3, 2.7)
XLIM     = (0, 40)          # n_IDR-DNA contacts
YLIM     = (0, 20)          # kJ/mol
STD_CEIL = 15.0             # kJ/mol — reliability truncation (as Panel D1)
# Δ2-10's n_IDR-DNA counts CA+CB of 11 residues (WT 1, 11-20), not the 20 of holo/IDR,
# so its x-scale is not directly comparable: shown dash-dot, as in D1-Δ2-10.
# system stem in analysis/<sys>_pmf_nidr_mean_std.dat -> style (colours as D1-Δ2-10)
SYS      = {"1Nhp6A_wall": dict(label="Nhp6A:DNA", color="#D55E00", ls="-"),
            "1IDR_wall":   dict(label="IDR:DNA",   color="#000000", ls="--"),
            "1D2_10_wall": dict(label="Δ2-10:DNA", color="#882255", ls="-.")}
TITLE    = ""
OUT      = "1Nhp6A_wall_pmf_nidr_vs_idr_d2_10"
plt.rcParams.update({"font.size": 12, "axes.labelsize": 10, "xtick.labelsize": 10,
                     "ytick.labelsize": 10, "legend.fontsize": 10, "axes.linewidth": 0.8,
                     "figure.dpi": 150, "axes.grid": False, "pdf.fonttype": 42})

fig, ax = plt.subplots(figsize=FIGSIZE)
for name, s in SYS.items():
    d = np.loadtxt(f"analysis/{name}_pmf_nidr_mean_std.dat")
    x, m, sd = d[:, 0], d[:, 1], d[:, 2]
    ok = f5.reliable_1d(m, sd, STD_CEIL)
    ax.plot(x[ok], m[ok], lw=2, color=s["color"], ls=s["ls"], label=s["label"])
    ax.fill_between(x[ok], (m - sd)[ok], (m + sd)[ok], color=s["color"], alpha=0.25, lw=0)
    print(f"  {s['label']:11s} minimum at {x[ok][np.nanargmin(m[ok])]:5.1f} contacts"
          f"   (reliable {x[ok].min():.1f} – {x[ok].max():.1f})")

ax.set_xlabel(r"$n_\mathrm{IDR-DNA}$ contacts")
ax.set_ylabel(r"$\Delta F$ (kJ mol$^{-1}$)")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.legend(frameon=False, loc="upper center", fontsize=9)
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

### Panel D2 — end-to-end distance, free DNA vs bound

$d_\mathrm{e-e}$ is the *biased* CV in both runs, so this is the best-determined comparison of the
two; the bend angle above is a spectator reweighted onto it.

In [ ]:
# ─── Panel D2: everything this figure needs ───────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — D2
VERSION  = "v2"          # v2: free-DNA reference rebuilt on DNA_wall2 (corrected dna_end2)
FIGSIZE  = (3.3, 2.7)
XLIM     = (3.0, 5.6)       # nm
YLIM     = (0, 20)          # kJ/mol
STD_CEIL = 15.0
SYS      = {"DNA":  dict(label="DNA alone", color="#0072B2"),
            "holo": dict(label="Nhp6A:DNA", color="#D55E00")}
TITLE    = ""
OUT      = "1Nhp6A_wall_pmf_de2e_vs_dna"
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})

fig, ax = plt.subplots(figsize=FIGSIZE)
for name, s in SYS.items():
    d = f5.load1d(f"D_pmf_de2e_{name}.dat")
    x, m, sd = d[list(d)[0]], d["F_mean"], d["F_std"]
    ok = f5.reliable_1d(m, sd, STD_CEIL)
    ax.plot(x[ok], m[ok], lw=2, color=s["color"], label=s["label"])
    ax.fill_between(x[ok], (m - sd)[ok], (m + sd)[ok], color=s["color"], alpha=0.25, lw=0)
    print(f"  {s['label']:11s} minimum at {x[ok][np.nanargmin(m[ok])]:5.2f} nm"
          f"   (reliable {x[ok].min():.2f} – {x[ok].max():.2f})")

ax.set_xlabel(r"$d_\mathrm{e-e}$ (nm)")
ax.set_ylabel(r"$\Delta F$ (kJ mol$^{-1}$)")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.legend(frameon=False, loc="upper left")
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

### Panel E — $F(\mathrm{bend},\, n_\mathrm{IDR-DNA})$

*"bending is possible only upon formation of IDR–DNA contacts, suggesting a decisive role for the
protein IDR."*

DNA bend angle on the x-axis, $n_\mathrm{IDR-DNA}$ on the y-axis — so the bending coordinate reads
left-to-right, consistent with panels D1/D2.

In [ ]:
# ─── Panel E: everything this figure needs ────────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — E
VERSION    = "v2"            # v2: transposed -> bend angle on x, n_IDR-DNA on y
FIGSIZE    = (4.2, 3.6)
XLIM       = (0, 60)         # DNA bend angle (deg) — matched to panels D1/D2
YLIM       = None            # n_IDR-DNA
VMAX       = 50              # kJ/mol colour ceiling
CMAP       = "RdYlBu_r"
STD_CEIL   = 15.0
MIN_VISITS = 5
TITLE      = ""
OUT        = "1Nhp6A_wall_pmf2d_idrdna_bend"

plt.rcParams.update({"font.size": 15, "axes.labelsize": 12, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "axes.linewidth": 0.8, "figure.dpi": 110,
                     "axes.grid": False, "pdf.fonttype": 42})

nc, bend, mean, std, cnt = f5.load2d("E_pmf2d_idrdna_bend.dat")
F = f5.reliable_2d(mean, std, cnt, STD_CEIL, MIN_VISITS)   # F indexed [nc, bend]

fig, ax = plt.subplots(figsize=FIGSIZE)
Fm = np.ma.masked_invalid(F)
# transposed axes: bend on x, n_IDR-DNA on y. contourf(X, Y, Z) wants Z shape (len(Y), len(X)),
# i.e. (len(nc), len(bend)) = F itself (not F.T).
im = ax.contourf(bend, nc, Fm, levels=np.linspace(0, VMAX, 21), cmap=CMAP, extend="max")
ax.contour(bend, nc, Fm, levels=np.arange(5, VMAX, 5), colors="k", linewidths=0.35, alpha=0.45)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$\Delta F$ (kJ mol$^{-1}$)")
cb.ax.tick_params(labelsize=8)
ax.set_box_aspect(1)            # square AXES box; the colourbar takes its space from the
                                # figure, not from the box (set_aspect would mix CV units)
ax.set_xlabel("DNA bend angle (deg)")
ax.set_ylabel(r"$n_\mathrm{IDR-DNA}$")
if XLIM:
    ax.set_xlim(*XLIM)
if YLIM:
    ax.set_ylim(*YLIM)
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

# quantify the claim: how far can the DNA bend at low vs high IDR-DNA contact number?
for lab, sl in [("few IDR–DNA contacts (<10)", nc < 10), ("many IDR–DNA contacts (>=20)", nc >= 20)]:
    col = np.nanmin(np.where(np.isfinite(F[sl]), F[sl], np.nan), axis=0)
    if np.isfinite(col).any():
        print(f"  {lab:28s}: minimum at {bend[np.nanargmin(col)]:.0f} deg, "
              f"resolved out to {bend[np.isfinite(col)].max():.0f} deg")

### Panel E-IDR — IDR-only $F(\mathrm{bend},\, n_\mathrm{IDR-DNA})$ with state medoids

The IDR-only analog of Panel E, on the same axes, grid, colour scale and truncation, so the two
maps can be set side by side. The DNA stays straight across the whole surface; what varies is how
much of the IDR is engaged.

**Two shallow basins** at 400 ns/walker (`select_snapshots.py --system 1IDR_wall_bend --barrier 3.0
--f-ceil 8.0 --min-cells 6 --t-max-ns 400 --n-per-state 200 --medoid-cv-tiebreak`; the same 2 basins
at every barrier from 1.0 to 3.0 kJ/mol, with or without `--min-cells`):

| # | $n_\mathrm{IDR-DNA}$ | bend | $F_\mathrm{min}$ | IDR engagement |
|---|---|---|---|---|
| 1 | ~12 | 8° | 0.00 | partial |
| 2 | ~26 | 8° | 0.20 | fully wrapped |

At 300 ns the same surface needed `--barrier 1.0` to give 3 basins (n~11/8°, n~26/8°, n~22/13°);
with the extra 100 ns the n~22/13° well is no longer separated from basin 2, so it is one basin now
(user decision 2026-11-03: 2 basins). The engagement labels carry over from the 300 ns per-residue
contact analysis of the matching basins (300 ns basin 1: anchored by R10/R13/K16 + C-terminal
K22/R23, residues 1–4 mostly off; 300 ns basins 2+3: residues 1–14 ≈ all in contact) — not yet
recomputed on the 400 ns medoid frames. **Caveats:** the two wells are 0.2 kJ/mol apart and each is
structurally heterogeneous (mean peptide RMSD to the medoid ~12 Å after superposing on the DNA), so
a medoid shows the typical placement, not a unique pose. Medoids are ranked by the peptide's RMSD
after superposition on the DNA (`medoid_fit`), not all-heavy-atom RMSD, which the DNA would decide.
PDBs: `analysis/snapshots/1IDR_wall_bend/state_<k>_medoid.pdb` (300 ns 3-basin set in
`analysis/backup_1IDR_wall_300ns/snapshots/`).

Masking is exactly that of Panel E (std ≤ `STD_CEIL`, ≥ `MIN_VISITS` visits; `MIN_WALKERS = None`),
so the two panels show the same reliability criterion. The medoid segmentation itself used the SI
ergodicity mask (≥3 of 6 walkers per bin); set `MIN_WALKERS = 3` to draw the map on that mask.

In [ ]:
# ─── Panel E-IDR: everything this figure needs ────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — E-IDR (matched to Panel E v2)
VERSION     = "v2"             # v2: 400 ns/walker, 2 basin medoids (barrier 3.0); v1: 300 ns, 3 basins (barrier 1.0)
FIGSIZE     = (4.2, 3.6)
XLIM        = (0, 60)        # DNA bend angle (deg) — matched to D1 and Panel E
YLIM        = None           # n_IDR-DNA: auto (grid extent), exactly as Panel E, so panels align
VMAX        = 50             # kJ/mol colour ceiling
CMAP        = "RdYlBu_r"
STD_CEIL    = 15.0
MIN_VISITS  = 5
MIN_WALKERS = None           # None = Panel E masking exactly (std + visits only); 3 = SI ergodicity mask
MEDOIDS     = "1IDR_wall_bend_medoids_cvs.dat"     # analysis/snapshots/
TITLE       = ""
OUT         = "1IDR_wall_pmf2d_idrdna_bend"

plt.rcParams.update({"font.size": 15, "axes.labelsize": 12, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "axes.linewidth": 0.8, "figure.dpi": 110,
                     "axes.grid": False, "pdf.fonttype": 42})

nc, bend, mean, std, cnt, cov = f5.load2d("1IDR_wall_pmf2d_idrdna_bend.dat",
                                          data="analysis", with_cov=True)
F = f5.reliable_2d(mean, std, cnt, STD_CEIL, MIN_VISITS,
                   walker_cov=cov, min_walkers=MIN_WALKERS)      # F indexed [nc, bend]

fig, ax = plt.subplots(figsize=FIGSIZE)
Fm = np.ma.masked_invalid(F)
im = ax.contourf(bend, nc, Fm, levels=np.linspace(0, VMAX, 21), cmap=CMAP, extend="max")
ax.contour(bend, nc, Fm, levels=np.arange(5, VMAX, 5), colors="k", linewidths=0.35, alpha=0.45)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$\Delta F$ (kJ mol$^{-1}$)")
cb.ax.tick_params(labelsize=8)

state, m_nc, m_bend, _ = f5.load_medoids(MEDOIDS, data="analysis/snapshots",
                                         cv1="n_IDR-DNA", cv2="bend")
for k, y, x in zip(state, m_nc, m_bend):
    ax.plot(x, y, "o", ms=13, mfc="white", mec="k", mew=1.0)
    ax.text(x, y, str(k), ha="center", va="center", fontsize=9, fontweight="bold")
    print(f"  medoid {k}: bend {x:5.1f} deg, n_IDR-DNA {y:5.1f}")

ax.set_box_aspect(1)
ax.set_xlabel("DNA bend angle (deg)")
ax.set_ylabel(r"$n_\mathrm{IDR-DNA}$")
ax.set_xlim(*XLIM)
if YLIM:
    ax.set_ylim(*YLIM)
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

i, j = np.unravel_index(np.nanargmin(np.where(np.isfinite(F), F, np.inf)), F.shape)
print(f"  global minimum: bend {bend[j]:.0f} deg, n_IDR-DNA {nc[i]:.1f}  "
      f"(reliable bins {np.isfinite(F).sum()})")
for lab, sl in [("few IDR–DNA contacts (<10)", nc < 10), ("many IDR–DNA contacts (>=20)", nc >= 20)]:
    col = np.nanmin(np.where(np.isfinite(F[sl]), F[sl], np.nan), axis=0)
    if np.isfinite(col).any():
        print(f"  {lab:28s}: minimum at {bend[np.nanargmin(col)]:.0f} deg, "
              f"resolved out to {bend[np.isfinite(col)].max():.0f} deg")

### Panel E-Δ2-10 — Δ2-10 $F(\mathrm{bend},\, n_\mathrm{IDR-DNA})$ with state medoids

The Δ2-10 analog of Panel E and E-IDR: same axes, grid, colour scale and masking as Panel E
(std ≤ `STD_CEIL`, ≥ `MIN_VISITS` visits), so the three maps align side by side.

**Full run, 350 ns/walker** (job 3918 complete). **The reweighting gate still fails** (ESS 19 %,
MAD(de2e) 4.30 kJ/mol; was 8 % / 5.04 at ~230 ns), so read this surface as less converged than WT or
IDR-only.

**$n_\mathrm{IDR-DNA}$ is a different CV here:** it counts CA+CB of 11 residues (new 1–11 = WT 1, 11–20)
instead of WT's 20, so equal heights on the y axis do not mean equal IDR engagement.

**Three basins** (`select_snapshots.py --system 1D2_10_wall_bend --barrier 3.0 --f-ceil 8.0
--min-cells 5 --t-max-ns 350 --n-per-state 200 --medoid-cv-tiebreak`; pool = canonical `_proc.xtc`,
20 ps; segmented on the SI ergodicity mask std ≤ 5, ≥3 walkers, as for WT):

| # | $n_\mathrm{IDR-DNA}$ | bend | $F_\mathrm{min}$ (kJ/mol) | note |
|---|---|---|---|---|
| 1 | ~1 | 28° | 1.20 | IDR detached; 189/200 frames from walker r5 alone — a single-walker excursion, not the global minimum |
| 2 | ~11 | 26° | 2.13 | 5 walkers; suggested representative |
| 3 | ~17 | 33° | 3.33 | 6 walkers |

The F = 0 cell (n 1.2, 24°) is visited by only 2 walkers. Basin populations are not resolved
(b2 nominally 66 %, but its range overlaps b3's; LABNOTEBOOK 2026-11-05). Medoids are all-heavy-atom
RMSD medoids (as WT; the protein dominates), PDBs in
`analysis/snapshots/1D2_10_wall_bend/state_<k>_medoid.pdb`; the preliminary 300 ns set is in
`_build/1D2_10_prelim_289ns/`.


In [ ]:
# ─── Panel E-Δ2-10: everything this figure needs ──────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — E-Δ2-10 (matched to Panel E v2)
VERSION     = "v2"             # v2: run complete, 350 ns/walker, canonical _proc.xtc pool (20 ps); v1: preliminary 300 ns, _prelim.xtc 0-295 ns
FIGSIZE     = (4.2, 3.6)
XLIM        = (0, 60)        # DNA bend angle (deg) — matched to D1 and Panel E
YLIM        = None           # n_IDR-DNA: auto (grid extent), exactly as Panel E, so panels align
VMAX        = 50             # kJ/mol colour ceiling
CMAP        = "RdYlBu_r"
STD_CEIL    = 15.0
MIN_VISITS  = 5
MIN_WALKERS = None           # None = Panel E masking exactly (std + visits only); 3 = SI ergodicity mask
MEDOIDS     = "1D2_10_wall_bend_medoids_cvs.dat"     # analysis/snapshots/
TITLE       = ""
OUT         = "1D2_10_wall_pmf2d_idrdna_bend"

plt.rcParams.update({"font.size": 15, "axes.labelsize": 12, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "axes.linewidth": 0.8, "figure.dpi": 110,
                     "axes.grid": False, "pdf.fonttype": 42})

nc, bend, mean, std, cnt, cov = f5.load2d("1D2_10_wall_pmf2d_idrdna_bend.dat",
                                          data="analysis", with_cov=True)
F = f5.reliable_2d(mean, std, cnt, STD_CEIL, MIN_VISITS,
                   walker_cov=cov, min_walkers=MIN_WALKERS)      # F indexed [nc, bend]

fig, ax = plt.subplots(figsize=FIGSIZE)
Fm = np.ma.masked_invalid(F)
im = ax.contourf(bend, nc, Fm, levels=np.linspace(0, VMAX, 21), cmap=CMAP, extend="max")
ax.contour(bend, nc, Fm, levels=np.arange(5, VMAX, 5), colors="k", linewidths=0.35, alpha=0.45)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label(r"$\Delta F$ (kJ mol$^{-1}$)")
cb.ax.tick_params(labelsize=8)

state, m_nc, m_bend, _ = f5.load_medoids(MEDOIDS, data="analysis/snapshots",
                                         cv1="n_IDR-DNA", cv2="bend")
for k, y, x in zip(state, m_nc, m_bend):
    ax.plot(x, y, "o", ms=13, mfc="white", mec="k", mew=1.0, clip_on=False, zorder=5)   # medoid 1 sits on the n=0.4 spine
    ax.text(x, y, str(k), ha="center", va="center", fontsize=9, fontweight="bold", clip_on=False, zorder=6)
    print(f"  medoid {k}: bend {x:5.1f} deg, n_IDR-DNA {y:5.1f}")

ax.set_box_aspect(1)
ax.set_xlabel("DNA bend angle (deg)")
ax.set_ylabel(r"$n_\mathrm{IDR-DNA}$")
ax.set_xlim(*XLIM)
if YLIM:
    ax.set_ylim(*YLIM)
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

i, j = np.unravel_index(np.nanargmin(np.where(np.isfinite(F), F, np.inf)), F.shape)
print(f"  global minimum: bend {bend[j]:.0f} deg, n_IDR-DNA {nc[i]:.1f}  "
      f"(reliable bins {np.isfinite(F).sum()})")
for lab, sl in [("few IDR–DNA contacts (<10)", nc < 10), ("many IDR–DNA contacts (>=20)", nc >= 20)]:
    col = np.nanmin(np.where(np.isfinite(F[sl]), F[sl], np.nan), axis=0)
    if np.isfinite(col).any():
        print(f"  {lab:28s}: minimum at {bend[np.nanargmin(col)]:.0f} deg, "
              f"resolved out to {bend[np.isfinite(col)].max():.0f} deg")

## 2:1 Nhp6A:DNA — H3 distortion and limited bending (panels F1/F2)

Unbiased equilibrium MD only (3 systems × 3 replicates × 250 ns, rubisco campaign).

### Panel F1 — the 2:1 complex: H3 distortion

*"Already in the short equilibrium simulations we find that the complex experiences large
distortions, particularly in helix H3."*

**Unbiased equilibrium MD only.** The 2Nhp6A metadynamics run fails its reweighting gate
(ESS ≈ 0 %, MAD = 4.7 kJ/mol), so no free energy may be quoted from it. H3 is Pro64–Ala93 (1LWM).
A single observable is shown — the **H3 backbone RMSD** from the reference — which captures the
distortion directly (the earlier helicity and kink-angle traces are dropped: helicity alone is blind
to the hinge motion, and RMSD already reports it).

In [ ]:
# ─── Panel F1: everything this figure needs ───────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — F1
VERSION    = "v2"          # v2: RMSD only (one panel), dropped helicity/kink subplots
OBS        = ["H3_RMSD_nm"]                                # RMSD is the H3-distortion observable kept
LABEL      = {"H3_helicity": "H3 helicity", "H3_RMSD_nm": "H3 RMSD (nm)",
              "H3_kink_deg": "H3 kink (deg)", "H3_displacement_nm": "H3 displacement (nm)",
              "HMG_helicity": "HMG helicity"}
COLOR      = {"A": "#4C72B0", "B": "#C44E52"}               # the two protein copies
FIGSIZE    = (3.9, 2.7)
N_BLOCKS   = 150            # block average drawn over the raw trace, for readability
TITLE      = ""
OUT        = "2Nhp6A_equil_h3_distortion"
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})

f1 = f5.load1d("F1_equil_2Nhp6A_h3.dat")
t = f1["time_ns"]
copies = sorted({k.rsplit("_", 1)[1] for k in f1 if k != "time_ns"})

fig, axes = plt.subplots(len(OBS), 1, figsize=FIGSIZE, sharex=True)
for ax, key in zip(np.atleast_1d(axes), OBS):
    for cp in copies:
        y = f1[f"{key}_{cp}"]
        ax.plot(t, y, lw=0.4, alpha=0.30, color=COLOR[cp])
        k = max(1, len(y) // N_BLOCKS)
        ax.plot(t[:len(y) // k * k].reshape(-1, k).mean(1),
                y[:len(y) // k * k].reshape(-1, k).mean(1),
                lw=1.4, color=COLOR[cp], label=f"copy {cp}")
        print(f"  copy {cp}  {LABEL[key]:18s} {y.mean():7.3f} +/- {y.std():.3f}")
    ax.set_ylabel(LABEL[key])
    ax.set_xlim(0, t[-1])          # time series: 0 -> final time, always
np.atleast_1d(axes)[0].legend(frameon=False, ncol=len(copies), loc="lower left")
np.atleast_1d(axes)[-1].set_xlabel("time (ns)")
if TITLE:
    np.atleast_1d(axes)[0].set_title(TITLE)
fig.align_ylabels(np.atleast_1d(axes))
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

In [ ]:
# ─── Panel F1 — replicate 2 (PBC-clean) ───────────────────────────────────────
# The default F1 above is rep0, which is PBC-flagged (copy-B H3 swing coincides with a marginal
# min-image contact). rep2 has no such artifact. Rendered through the same f5.f1_panel() helper
# so it is directly comparable to F1 above (identical observable, axes, colours, block averaging).
# Use it to decide which replicate the paper shows. rep2 still shows the H3 distortion of copy B
# (mean 0.33 vs copy A 0.18 nm), milder than rep0's 0.48; rep1 (0.23) is milder still -- the
# distortion magnitude is replicate-dependent.
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import matplotlib.pyplot as plt
import article_plot as f5
FIGSIZE  = (3.3, 1.5)
VERSION = "v1"          # v1: first render of the rep2 variant
OUT     = "2Nhp6A_equil_h3_distortion_rep2"
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})

stats = f5.f1_panel("F1_equil_2Nhp6A_h3_rep2.dat", out=OUT, version=VERSION, figs=f5.ARTICLE,
                    obs=("H3_RMSD_nm",), title="", figsize=FIGSIZE)
for (obs, cp), (mean, sd) in stats.items():
    print(f"  copy {cp}  {obs:12s} {mean:7.3f} +/- {sd:.3f}")
plt.show()

### Panel F2 — bending in the 2:1 complex

*"Due to the large steric hindrance between the two Nhp6As, the degree of bending does not reach
that of the 1:1 complex."*

The honest comparison is **unbiased vs unbiased**: equilibrium bend-angle distributions. The
equilibrium campaign is still running round-robin, so this renders whatever replicas exist and says
which are missing. (Comparing the trimer's *biased* distribution against a *reweighted* 1:1 PMF
would not be a valid comparison, and is deliberately not done.)

In [ ]:
# ─── Panel F2: everything this figure needs ───────────────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — F2
VERSION = "v2"          # v2: all 3 systems x 3 reps (v1 had a path bug: 2Nhp6A rep0 only)
FIGSIZE = (3.3, 2.7)
XLIM    = (0, 60)        # degrees
REBIN   = 3              # the .dat histogram has 180 bins; coarsen for display
SYS     = {"2Nhp6A":      ("2Nhp6A:DNA", "#009E73"),
           "1Nhp6A_wall": ("Nhp6A:DNA",  "#D55E00"),
           "DNA_wall":    ("DNA alone",  "#0072B2")}
TITLE   = ""
OUT     = "2Nhp6A_equil_bend_compare"
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})

f2 = f5.load1d("F2_equil_bend_hist.dat")
x = f2["bend_deg"]
present = [s for s in SYS if f"counts_{s}" in f2]
missing = [s for s in SYS if s not in present]

fig, ax = plt.subplots(figsize=FIGSIZE)
for s in present:
    lab, col = SYS[s]
    c = f2[f"counts_{s}"]
    n = len(c) // REBIN * REBIN
    cc = c[:n].reshape(-1, REBIN).sum(1)
    xx = x[:n].reshape(-1, REBIN).mean(1)
    dens = cc / (cc.sum() * (xx[1] - xx[0]))
    ax.plot(xx, dens, lw=1.8, color=col, label=lab)
    ax.fill_between(xx, 0, dens, color=col, alpha=0.15)
    print(f"  {lab:12s} mean bend = {np.sum(xx * cc) / cc.sum():.1f} deg  ({cc.sum():,} frames)")

ax.set_xlabel("DNA bend angle (deg)")
ax.set_ylabel("probability density")
ax.set_xlim(*XLIM)
ax.legend(frameon=False)
if TITLE:
    ax.set_title(TITLE)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()

if missing:
    print("\nPARTIAL — still missing equilibrium data for: " + ", ".join(missing))
    print("Re-run article_data.py --panels F when the campaign delivers them.")

## SI — MM distortion and the m / mM probes (pulled starts)

Supports the main-text section *"Insights on MM and m-states from atomistic MD"*. **Unbiased MD only**
(no weights): time series and last-window statistics, never populations. Tables from
`article_data.py --panels M` (`analysis/article/M_*.dat`); bound-mode labels from
`analysis/article/M_mstate_1to1_modes.dat` (240–250 ns, best-register native core–DNA Q).
All m-state runs are shown over **250 ns** (the two 500-ns extensions are cut, so every run has the same length).
Structure snapshots are rendered separately and placed in `figures/article/figure_si_mstate.svg`.

- **MA** 2:1 from the AF3 MM model: H3 RMSD per copy, all three replicates (distortion is replicate-dependent).
- **MB** 1:1 m-state: core–DNA COM distance after release (8 runs × 250 ns).
- **MC** 2:1 mM probe: copy-B core re-approach; copy A stays bound.
- **MD** core contact PC1 (common native-relative frame) vs distance: separates native from
  non-native poses at the same distance. PC1 is register-specific, so the −2 bp shifted run
  (c1_rep1) sits low on PC1 although its contacts are native-like in another register.
- **ME** 1:1 m-state: core–DNA Q in the **native register only** (companion of MB, same colours).
- **MF** 2:1 mM probe: **best-register** core–DNA Q per copy (companion of MC, same colours);
  copy B's site in the 2:1 complex is not fixed, so every register variant is allowed.

In [ ]:
# ─── Panel MA — 2:1 (MM start) H3 distortion, all three replicates ────────────
# Same observable, colours and block averaging as Panel F1 (H3 = 1LWM Pro64-Ala93, backbone RMSD
# to the run's frame 0 = the equilibrated AF3 2:1 model), one row per replicate so the
# replicate dependence is visible (copy B mean 0.48 / 0.23 / 0.33 nm in rep0/1/2).
# rep0: copy B's H3 C-tip transiently approaches its periodic image (min 0.553 nm, < 1.2 nm for 2.9%
# of frames; LABNOTEBOOK 2026-07-30). Not labelled in the panel (user): stated in the caption instead.
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})
VERSION  = "v1"          # v1: first render
OUT      = "2Nhp6A_equil_h3_rmsd_allreps"
REPS     = (0, 1, 2)
COLOR    = f5.F1_COLOR                     # copy A blue, copy B red
N_BLOCKS = 150
FIGSIZE  = (2.4, 4.6)

fig, axes = plt.subplots(len(REPS), 1, figsize=FIGSIZE, sharex=True, sharey=True)
for ax, rep in zip(axes, REPS):
    d = f5.load1d(f"F1_equil_2Nhp6A_h3_rep{rep}.dat")
    t = d["time_ns"]
    for cp in "AB":
        y = d[f"H3_RMSD_nm_{cp}"]
        ax.plot(t, y, lw=0.4, alpha=0.30, color=COLOR[cp])
        k = max(1, len(y) // N_BLOCKS)
        n = len(y) // k * k
        ax.plot(t[:n].reshape(-1, k).mean(1), y[:n].reshape(-1, k).mean(1),
                lw=1.4, color=COLOR[cp], label=f"copy {cp}")
        print(f"  rep{rep} copy {cp}  H3 RMSD {y.mean():.3f} +/- {y.std():.3f} nm")
    ax.text(0.02, 0.92, f"rep {rep}", transform=ax.transAxes,
            va="top", fontsize=8)
    ax.set_xlim(0, t[-1])          # time series: 0 -> final time, always
    ax.set_ylim(0, None)
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, ncol=2,
           loc="upper center", bbox_to_anchor=(0.55, 0.985))
axes[len(REPS) // 2].set_ylabel("H3 RMSD (nm)")
axes[-1].set_xlabel("time (ns)")
fig.tight_layout(h_pad=0.3, rect=(0, 0, 1, 0.95))
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()


In [ ]:
# ─── Panel MB — 1:1 m-state: core-DNA distance after release from the pulled start ─
# d_core = |COM(Calpha 27-93) - COM(DNA)|, the pulled distance itself (PLUMED monitor colvar,
# 0.1 ns samples, drawn as 1-ns block means). 4 configs x 2 seeds x 250 ns (all tables cut at
# 250 ns by article_data.py). Colour = bound mode over 240-250 ns (M_mstate_1to1_modes.dat:
# best register-variant native core-DNA Q >= 0.5): native / shifted register / non-native. Grey band = unbiased 1:1 holo mean +/- 2 SD
# (3 x 250 ns, t >= 20 ns). Unbiased MD from imposed starts: lifetimes, not populations.
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})
VERSION  = "v1"          # v1: first render
OUT      = "1Nhp6A_mstate_dcore"
BLOCK_NS = 1.0
T_MAX    = 250.0        # ns: every run (tables are already cut here)
INSET_NS = 30.0          # zoom on the release / first re-contact
MODE_COLOR = {"native": "#2A9D8F", "shifted": "#E9A03B", "non-native": "#7F7F7F"}
FIGSIZE  = (3.2, 2.2)

d = f5.load1d("M_mstate_1to1_timeseries.dat")
holo = f5.load1d("M_holo_dcore.dat")
th = holo["time_ns"]
band = np.concatenate([holo[f"d_core_1Nhp6A_holo_rep{r}_A"][th >= 20] for r in range(3)])
lo, hi = band.mean() - 2 * band.std(), band.mean() + 2 * band.std()   # mean +/- 2 SD (user)
mode = {}
for line in open("analysis/article/M_mstate_1to1_modes.dat"):
    if not line.startswith("#") and line.strip():
        f = line.split()
        mode[f[0]] = (f[1], f[2])              # mode, register
runs = sorted({k.rsplit("_", 2)[-2] + "_" + k.rsplit("_", 1)[-1] for k in d if k.startswith("d_core_")})

def blocks(t, y, w):
    ok = np.isfinite(y)
    t, y = t[ok], y[ok]
    k = max(1, int(round(w / (t[1] - t[0]))))
    n = len(y) // k * k
    return t[:n].reshape(-1, k).mean(1), y[:n].reshape(-1, k).mean(1)

fig, ax = plt.subplots(figsize=FIGSIZE)
ins = ax.inset_axes([0.52, 0.60, 0.45, 0.36])
for a in (ax, ins):
    a.axhspan(lo, hi, color="0.85", lw=0, zorder=0)
t = d["time_ns"]
for run in runs:
    y = d[f"d_core_{run}"]
    m, reg = mode[run]
    c = MODE_COLOR[m]
    tb, yb = blocks(t[t <= T_MAX], y[t <= T_MAX], BLOCK_NS)
    ax.plot(tb, yb, lw=0.9, color=c)
    z = t <= INSET_NS
    ins.plot(t[z], y[z], lw=0.6, color=c)
    tend = t[np.isfinite(y)][-1]
    print(f"  {run}: t0 {y[0]:.2f} nm, last-10-ns {np.nanmean(y[(t > tend - 10) & (t <= tend)]):.2f} nm, "
          f"{tend:.0f} ns, {m} ({reg})")
ax.set_xlim(0, T_MAX)
ax.set_ylim(1.2, 4.6)
ins.set_xlim(0, INSET_NS)
ins.set_ylim(1.2, 3.9)
ins.tick_params(labelsize=7)
ax.set_xlabel("time (ns)")
ax.set_ylabel(r"$d_\mathrm{core-DNA}$ (nm)")
for m, c in MODE_COLOR.items():
    ax.plot([], [], color=c, lw=1.4, label=f"{m} ({sum(mode[r][0] == m for r in runs)})")
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0.0, 0.72), fontsize=7)
print(f"  holo 1:1 d_core mean +/- 2 SD {lo:.2f}-{hi:.2f} nm")
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()


In [ ]:
# ─── Panel MC — 2:1 mM probe: copy-B core re-approach, copy A stays bound ──────
# Same d_core definition as Panel MB, per protein copy (PLUMED d_core_A / d_core_B). Copy B was
# pulled off from the 2:1 holo run (rep1 frames), copy A left unbiased. Bands = unbiased 2:1
# holo mean +/- 2 SD of EACH copy (reps 1-2, t >= 20 ns; rep0 excluded as in the start-frame selection).
# The 4 runs are drawn alike (copy A blue, copy B red). Note for the caption: both c1 starts carry an
# open A12.B4 pair under copy A's wedge, inherited from the pull.
# Copy B is non-native in all four runs over the last 10 ns (2Nhp6A_mstate_md_stageb_modes_v1.dat).
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})
VERSION  = "v1"          # v1: first render
OUT      = "2Nhp6A_mstate_dcore"
BLOCK_NS = 1.0
RUNS     = ["c1_rep0", "c1_rep1", "c3_rep0", "c3_rep1"]
COLOR    = f5.F1_COLOR       # copy A blue, copy B red: as in Panel MA; runs are not distinguished
FIGSIZE  = (3.2, 2.2)

d = f5.load1d("M_mstate_2to1_timeseries.dat")
holo = f5.load1d("M_holo_dcore.dat")
th = holo["time_ns"]
pooled = {cp: np.concatenate([holo[f"d_core_2Nhp6A_rep{r}_{cp}"][th >= 20] for r in (1, 2)]) for cp in "AB"}
bands = {cp: (x.mean() - 2 * x.std(), x.mean() + 2 * x.std()) for cp, x in pooled.items()}   # mean +/- 2 SD (user)

def blocks(t, y, w):
    ok = np.isfinite(y)
    t, y = t[ok], y[ok]
    k = max(1, int(round(w / (t[1] - t[0]))))
    n = len(y) // k * k
    return t[:n].reshape(-1, k).mean(1), y[:n].reshape(-1, k).mean(1)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.axhspan(*bands["A"], color=COLOR["A"], alpha=0.15, lw=0, zorder=0)
ax.axhspan(*bands["B"], color=COLOR["B"], alpha=0.15, lw=0, zorder=0)
t = d["time_ns"]
for run in RUNS:
    for cp in "AB":
        tb, yb = blocks(t, d[f"d_core_{cp}_{run}"], BLOCK_NS)
        ax.plot(tb, yb, lw=0.9, color=COLOR[cp], label=f"copy {cp}" if run == RUNS[0] else None)
    yB, yA = d[f"d_core_B_{run}"], d[f"d_core_A_{run}"]
    last = t > t[np.isfinite(yB)][-1] - 10
    print(f"  {run}: copy B t0 {yB[0]:.2f} -> last-10-ns {np.nanmean(yB[last]):.2f} nm; "
          f"copy A t0 {yA[0]:.2f} -> {np.nanmean(yA[last]):.2f} nm")
ax.set_xlim(0, max(t[np.isfinite(d[f"d_core_B_{r}"])][-1] for r in RUNS))   # 0 -> final time
ax.set_ylim(1.2, 4.6)
ax.set_xlabel("time (ns)")
ax.set_ylabel(r"$d_\mathrm{core-DNA}$ (nm)")
ax.legend(frameon=False, ncol=2, loc="upper right")
for cp, (lo, hi) in bands.items():
    print(f"  2:1 holo copy {cp} d_core mean +/- 2 SD {lo:.2f}-{hi:.2f} nm")
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()


In [ ]:
# ─── Panel MD — native-ness vs distance: core PC1 against d_core ───────────────
# PC1 of the core-only (residues 27-93) contact PCA in the common native-relative frame
# (2Nhp6A_mstate_md_mmcompare_v2.npz; each copy's contacts mapped onto the 1:1 site by its measured
# register: copy A native, copy B flip -2 bp). PC1 = native core interface (83% of its weight on
# the 30 native pairs). Clouds = unbiased holo (all frames, t >= 20 ns); points = pulled runs over
# their LAST WINDOW_NS (200-250 ns; all runs cut at 250 ns). Distance alone cannot separate native from non-native poses at ~1.6-1.7
# nm; PC1 can. No weights: point density is sampling, not population.
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})
VERSION   = "v1"         # v1: first render
OUT       = "mstate_pc1_dcore"
WINDOW_NS = 50.0
MODE_COLOR = {"native": "#2A9D8F", "shifted": "#E9A03B", "non-native": "#7F7F7F"}
FIGSIZE   = (3.6, 3.0)

import pandas as pd
df = pd.read_csv("analysis/article/M_pc1_dcore.dat", sep=r"\s+", comment="#",
                 names=["set", "run", "copy", "time_ns", "PC1", "Q_native", "Q_best", "d_core", "bend"],
                 dtype={"run": str})
mode = {}
for line in open("analysis/article/M_mstate_1to1_modes.dat"):
    if not line.startswith("#") and line.strip():
        f = line.split()
        mode[f[0]] = f[1]

fig, ax = plt.subplots(figsize=FIGSIZE)
h1 = df[(df.set == "1:1_holo") & (df.time_ns >= 20)]
hB = df[(df.set == "2:1_holo") & (df["copy"] == "B") & (df.time_ns >= 20)]
ax.scatter(h1.d_core, h1.PC1, s=1, color="0.75", alpha=0.25, lw=0, rasterized=True)
ax.scatter(hB.d_core, hB.PC1, s=1, color="#E8A6A8", alpha=0.25, lw=0, rasterized=True)
for run, g in df[df.set == "1:1_m-state"].groupby("run"):
    g = g[g.time_ns > g.time_ns.max() - WINDOW_NS]
    ax.scatter(g.d_core, g.PC1, s=2, color=MODE_COLOR[mode[run]], alpha=0.5, lw=0, rasterized=True)
    print(f"  1:1 {run:8s} last {WINDOW_NS:.0f} ns: d_core {g.d_core.mean():.2f}, PC1 {g.PC1.mean():5.2f} ({mode[run]})")
for run, g in df[(df.set == "2:1_m-state") & (df["copy"] == "B")].groupby("run"):
    g = g[g.time_ns > g.time_ns.max() - WINDOW_NS]
    ax.scatter(g.d_core, g.PC1, s=4, marker="^", color="#C44E52", alpha=0.6, lw=0, rasterized=True)
    print(f"  2:1 copy B {run:8s} last {WINDOW_NS:.0f} ns: d_core {g.d_core.mean():.2f}, PC1 {g.PC1.mean():5.2f}")
start = df[(df.time_ns == 0) & df.set.str.contains("m-state") & ~((df.set == "2:1_m-state") & (df["copy"] == "A"))]
ax.scatter(start.d_core, start.PC1, s=40, marker="*", color="k", lw=0, zorder=5)
# legend proxies
ax.scatter([], [], s=12, color="0.75", label="1:1 holo")
ax.scatter([], [], s=12, color="#E8A6A8", label="2:1 holo, copy B")
for m, c in MODE_COLOR.items():
    ax.scatter([], [], s=12, color=c, label=f"1:1 m-state, {m}")
ax.scatter([], [], s=14, marker="^", color="#C44E52", label="2:1 m-state, copy B")
ax.scatter([], [], s=40, marker="*", color="k", label="pulled starts")
ax.set_xlabel(r"$d_\mathrm{core-DNA}$ (nm)")
ax.set_ylabel("core contact PC1 (native →)")
ax.legend(frameon=False, fontsize=6.5, loc="upper right", markerscale=1.2)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()


In [ ]:
# ─── Panel ME — 1:1 m-state: native core-DNA Q after release (companion of Panel MB) ─
# Q_native = fraction of the 30 holo-native core(27-93)-DNA residue-nucleotide contacts formed, in
# the NATIVE register only (user: for the 1:1 complex, "the native state, period"); same switch and
# pair set as the bound-mode labels. Shifted-register runs therefore stay low here although their
# contacts are native-like on other bases. 0.1 ns samples drawn as 1-ns block means; all runs
# 250 ns. Colours = bound mode over 240-250 ns, exactly as in Panel MB. Grey band = unbiased 1:1
# holo mean +/- 2 SD (3 x 250 ns, t >= 20 ns).
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})
VERSION  = "v1"          # v1: first render
OUT      = "1Nhp6A_mstate_qnative"
BLOCK_NS = 1.0
T_MAX    = 250.0
MODE_COLOR = {"native": "#2A9D8F", "shifted": "#E9A03B", "non-native": "#7F7F7F"}
FIGSIZE  = (3.2, 2.2)

import pandas as pd
df = pd.read_csv("analysis/article/M_pc1_dcore.dat", sep=r"\s+", comment="#",
                 names=["set", "run", "copy", "time_ns", "PC1", "Q_native", "Q_best", "d_core", "bend"],
                 dtype={"run": str})
mode = {}
for line in open("analysis/article/M_mstate_1to1_modes.dat"):
    if not line.startswith("#") and line.strip():
        f = line.split()
        mode[f[0]] = (f[1], f[2])              # mode, register
h = df[(df.set == "1:1_holo") & (df.time_ns >= 20)].Q_native
lo, hi = h.mean() - 2 * h.std(), h.mean() + 2 * h.std()   # mean +/- 2 SD (as Panel MB)

def blocks(t, y, w):
    k = max(1, int(round(w / np.median(np.diff(t)))))
    n = len(y) // k * k
    return t[:n].reshape(-1, k).mean(1), y[:n].reshape(-1, k).mean(1)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.axhspan(lo, min(hi, 1.0), color="0.85", lw=0, zorder=0)
for run, g in df[df.set == "1:1_m-state"].groupby("run"):
    g = g.sort_values("time_ns")
    m, reg = mode[run]
    tb, yb = blocks(g.time_ns.to_numpy(), g.Q_native.to_numpy(), BLOCK_NS)
    ax.plot(tb, yb, lw=0.9, color=MODE_COLOR[m])
    print(f"  {run}: Q_native 240-250 ns {g.Q_native[g.time_ns >= 240].mean():.3f}  ({m}, {reg})")
for m, c in MODE_COLOR.items():
    ax.plot([], [], color=c, lw=1.4, label=f"{m} ({sum(v[0] == m for v in mode.values())})")
ax.set_xlim(0, T_MAX)
ax.set_ylim(0, 1)
ax.set_xlabel("time (ns)")
ax.set_ylabel(r"$Q_\mathrm{core-DNA}$")
#ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.18, 0.64), fontsize=7)
print(f"  holo 1:1 Q_native mean +/- 2 SD {lo:.2f}-{hi:.2f}")
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()


In [ ]:
# ─── Panel MF — 2:1 mM probe: best-register core-DNA Q per copy (companion of Panel MC) ─
# Q_best = per frame, the maximum over the 10 register variants (shifts -2..+2 bp, dyad flip) of the
# native core-DNA Q, in the common native-relative frame (each copy's contacts mapped onto the 1:1
# site by its measured register: copy A native, copy B flip -2 bp). Best register because copy B's
# binding site in the 2:1 complex is not fixed (user). Taking a per-frame maximum biases Q_best up
# slightly; for frames with no native-like contacts it stays near 0. Colours as in Panel MC
# (copy A blue, copy B red, runs not distinguished). Bands =
# unbiased 2:1 holo mean +/- 2 SD of the SAME quantity per copy (reps 1-2, t >= 20 ns).
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5
plt.rcParams.update({"font.size": 9, "axes.labelsize": 10, "xtick.labelsize": 9,
                     "ytick.labelsize": 9, "legend.fontsize": 8, "axes.linewidth": 0.8,
                     "figure.dpi": 110, "axes.grid": False, "pdf.fonttype": 42})
VERSION  = "v1"          # v1: first render
OUT      = "2Nhp6A_mstate_qbest"
BLOCK_NS = 1.0
RUNS     = ["c1_rep0", "c1_rep1", "c3_rep0", "c3_rep1"]
COLOR    = f5.F1_COLOR       # copy A blue, copy B red: as in Panel MA; runs are not distinguished
FIGSIZE  = (3.2, 2.2)

import pandas as pd
df = pd.read_csv("analysis/article/M_pc1_dcore.dat", sep=r"\s+", comment="#",
                 names=["set", "run", "copy", "time_ns", "PC1", "Q_native", "Q_best", "d_core", "bend"],
                 dtype={"run": str})

def blocks(t, y, w):
    k = max(1, int(round(w / np.median(np.diff(t)))))
    n = len(y) // k * k
    return t[:n].reshape(-1, k).mean(1), y[:n].reshape(-1, k).mean(1)

fig, ax = plt.subplots(figsize=FIGSIZE)
for cp, col in COLOR.items():
    h = df[(df.set == "2:1_holo") & (df["copy"] == cp) & (df.time_ns >= 20)].Q_best
    lo, hi = h.mean() - 2 * h.std(), h.mean() + 2 * h.std()
    ax.axhspan(lo, min(hi, 1.0), color=col, alpha=0.15, lw=0, zorder=0)
    print(f"  2:1 holo copy {cp} Q_best mean +/- 2 SD {lo:.2f}-{hi:.2f}")
ms = df[df.set == "2:1_m-state"]
for run in RUNS:
    gB = ms[(ms.run == run) & (ms["copy"] == "B")].sort_values("time_ns")
    gA = ms[(ms.run == run) & (ms["copy"] == "A")].sort_values("time_ns")
    for cp, g in (("A", gA), ("B", gB)):
        tb, yb = blocks(g.time_ns.to_numpy(), g.Q_best.to_numpy(), BLOCK_NS)
        ax.plot(tb, yb, lw=0.9, color=COLOR[cp], label=f"copy {cp}" if run == RUNS[0] else None)
    print(f"  {run}: last 10 ns Q_best copy B {gB.Q_best[gB.time_ns >= 240].mean():.3f}, "
          f"copy A {gA.Q_best[gA.time_ns >= 240].mean():.3f}")
ax.set_xlim(0, ms.time_ns.max())           # 0 -> final time
ax.set_ylim(0, 1)
ax.set_xlabel("time (ns)")
ax.set_ylabel(r"$Q_\mathrm{core-DNA}$")
ax.legend(frameon=False, ncol=2, loc="upper right")
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE)
plt.show()


## SI — phosphomimetic mutants (formerly `analysis_figure_supp.ipynb`)

### Mutant panels — provenance

Supporting-information panels for the Nhp6A:DNA metadynamics paper. This notebook
collects the SI material that extends the main-text 1:1 panels, starting with the
three point mutants (**S26D**, **T63D**, **TM**) run with the same soft-wall 2D
well-tempered MetaD protocol as WT holo (`coord_tail_dna` × `d_e2e`, 6 walkers,
all converged on the biased CVs).

Panels here reuse the article plotting conventions (`analysis/scripts/article_plot.py`):
the same reliability truncation (`reliable_1d` / `reliable_2d`), colour map, and the
project figure-naming rule. Every code cell is self-contained and re-runnable after a
kernel restart.

**Run lengths are not uniform.** WT, S26D and T63D were extended 300 → **350 ns/walker**
(jobs 9191–9193, all re-gated: converged, minima positionally unchanged). The TM extension
(9194) was still running when these panels were made, so **TM remains at 300 ns/walker** —
its curves are labelled accordingly in every panel. The extra 50 ns changed no minimum in
the three extended systems and made their under-sampled corners *denser* but not better
*mixed*, so the TM panels are not expected to move either; they are simply the shorter run.

**Data provenance** (generated once, then only loaded here):

| panel | table | produced by | run length |
|---|---|---|---|
| S1 — 2D $F(n_\mathrm{IDR-DNA}, \mathrm{bend})$ | `analysis/<sys>_pmf2d_idrdna_bend.dat` | `dna_bending_wall_compare.py --idrdna-bend-2d` | 350 ns (TM 300) |
| S2 — 1D $F(\mathrm{bend})$ overlay | `analysis/<sys>_pmf_bend_mean_std.dat` | `dna_bending_wall_compare.py` (run_all) | 350 ns (TM 300, DNA 100) |
| S3 — 1D $F(n_\mathrm{IDR-DNA})$ overlay | `analysis/<sys>_pmf_nidr_mean_std.dat` | `dna_bending_wall_compare.py` (run_all) | 350 ns (TM 300) |

All are final-bias reweighted with the same 5-snapshot mean±std ensemble used throughout
the project. `<sys>` ∈ {`DNA_wall2`, `1Nhp6A_wall`, `1S26D_wall`, `1T63D_wall`, `1TM_wall`}
for S1/S2; S3 has no `DNA_wall2` entry (`coord_tail_dna` is undefined for free DNA).

The 2D tables carry a 6th column, **`n_walkers`** — the number of the 6 walkers that
independently sample each bin (≥50 frames). It drives the ergodicity mask described under
panel S1.

### Panel S1 — 2D PMFs $F(n_\mathrm{IDR-DNA},\ \mathrm{bend})$ for WT and the three mutants

The direct analog of **Panel E**, one map per system on identical axes: DNA bend angle
($180-\theta$) on *x*, number of IDR–DNA contacts on *y*, $\Delta F$ colour-coded. Reliability
truncation and colour scale are shared with Panel E (`STD_CEIL = 15` kJ/mol,
`MIN_VISITS = 5`, `VMAX = 50` kJ/mol, `RdYlBu_r`).

**Ergodicity mask (`MIN_WALKERS = 3`).** In addition to the main-text sampling/std masks,
a bin is shown only where **at least 3 of the 6 walkers** independently sample it. A region
reached by just one or two walkers is not an equilibrated free-energy estimate even when
densely populated — its reweighted value still drifts in time — and a per-bin std ceiling
does *not* catch this, because two walkers that sit in a bin consistently give it a low
cross-snapshot std. This specifically removes the S26D low-bend / high-contact corner, traced
by a convergence audit to only **2 of 6 walkers**, one entering at 65 ns (see the caveat cell
below). The mask is off by default in `reliable_2d`, so the **main-text panels are
unaffected**; every global minimum here is preserved.

**The mask is specific to S26D, not a blanket penalty on the mutants.** Re-running the corner
audit at the current run lengths gives, for the corner `bend < 25°` / $n_\mathrm{IDR-DNA} > 25$,
the number of walkers with ≥1 % occupancy and the number of corner bins reached by ≥3 and by
≥4 walkers:

| system | walkers ≥1 % | bins ≥3 walkers | bins ≥4 walkers |
|---|---|---|---|
| WT (350 ns) | 3/6 | 23 | 12 |
| S26D (350 ns) | **2/6** | **20** | **1** |
| T63D (350 ns) | 4/6 | 49 | 33 |
| TM (300 ns) | 4/6 | 28 | 9 |

S26D is the outlier by an order of magnitude on the ≥4-walker count, and it is the only system
whose corner survives the ≥3 test almost entirely on two walkers' occupancy. Note also that the
+50 ns extension made every extended system's corner *denser* but not better *mixed* (WT and
T63D ≥4-walker bin counts unchanged over the extension) — more of the same sampling is not the
lever here, which is why TM's 300 ns is not the limiting factor for this panel.

In [ ]:
# ─── Panel S1: 2D F(n_IDR-DNA, bend), WT vs the three point mutants ────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — S1 (matched to Panel E)
VERSION     = "v3"          # v3: WT/S26D/T63D tables rebuilt at 350 ns (TM still 300 ns)
XLIM        = (0, 60)       # DNA bend angle (deg)
YLIM        = (0, 40)       # n_IDR-DNA contacts
VMAX        = 50           # kJ/mol colour ceiling
CMAP        = "RdYlBu_r"
STD_CEIL    = 50.0
MIN_VISITS  = 2
MIN_WALKERS = 2            # >=3 of 6 walkers must sample a bin (drops non-ergodic regions)
OUT         = "mutants_pmf2d_idrdna_bend"
# system -> panel title. WT first as the reference. Run length is carried in the title
# because it is NOT uniform: the 1TM extension (9194) had not landed when this was made.
PANELS = [("1Nhp6A_wall", "WT"),
          ("1S26D_wall",  "S26D"),
          ("1T63D_wall",  "T63D"),
          ("1TM_wall",    "TM")]

plt.rcParams.update({"font.size": 13, "axes.labelsize": 12, "xtick.labelsize": 11,
                     "ytick.labelsize": 11, "axes.linewidth": 0.8, "figure.dpi": 110,
                     "axes.grid": False, "pdf.fonttype": 42})

fig, axes = plt.subplots(2, 2, figsize=(8.2, 7.0), sharex=True, sharey=True)
levels = np.linspace(0, VMAX, 21)
im = None
for ax, (name, title) in zip(axes.ravel(), PANELS):
    nc, bend, mean, std, cnt, cov = f5.load2d(f"{name}_pmf2d_idrdna_bend.dat",
                                              data="analysis", with_cov=True)
    F = f5.reliable_2d(mean, std, cnt, STD_CEIL, MIN_VISITS,
                       walker_cov=cov, min_walkers=MIN_WALKERS)   # indexed [nc, bend]
    Fm = np.ma.masked_invalid(F)
    # transposed axes: bend on x, n_IDR-DNA on y -> contourf(X=bend, Y=nc, Z=F)
    im = ax.contourf(bend, nc, Fm, levels=levels, cmap=CMAP, extend="max")
    ax.contour(bend, nc, Fm, levels=np.arange(5, VMAX, 5), colors="k",
               linewidths=0.35, alpha=0.45)
    ax.set_title(title, fontsize=13)
    ax.set_xlim(*XLIM)
    if YLIM:
        ax.set_ylim(*YLIM)
    i, j = np.unravel_index(np.nanargmin(np.where(np.isfinite(F), F, np.inf)), F.shape)
    print(f"  {title:20s} basin: bend {bend[j]:4.0f} deg, n_IDR-DNA {nc[i]:4.1f}"
          f"   (reliable bins {np.isfinite(F).sum()})")

for ax in axes[-1, :]:
    ax.set_xlabel("DNA bend angle (deg)")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$n_\mathrm{IDR-DNA}$")

cb = fig.colorbar(im, ax=axes, pad=0.02, fraction=0.046, aspect=30)
cb.set_label(r"$\Delta F$ (kJ mol$^{-1}$)")
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE, transparent=False)
plt.show()

### Convergence caveat — the S26D low-bend / high-contact region

The white region at low bend and high $n_\mathrm{IDR-DNA}$ in the S26D panel is **masked on
purpose**, not missing data. A convergence audit of that corner (`bend < 25°`, $n_\mathrm{IDR-DNA} > 25$),
re-run at 350 ns and compared against the same run truncated at 300 ns, found:

- it is populated by **only 2 of the 6 walkers** (350 ns: wk0 28.6 %, wk4 37.3 %; the other four
  ≤ 0.4 %) — the same two as at 300 ns (31.0 % / 38.0 %), so the extension recruited no new walker;
- **wk4 enters only at 65 ns** (median residence 198 ns), so it accumulates that state in the
  second half of the run — cumulative-time snapshots therefore disagree there by construction;
- the cross-snapshot std **grows** when the snapshot count is doubled, the signature of genuine
  time-drift rather than finite-sample noise;
- the +50 ns changed nothing that matters: pooled corner occupancy 11.6 → 11.1 %, corner bins
  reached by ≥3 walkers **20 → 20**, by ≥4 walkers **0 → 1**. More of the same sampling is not
  the lever.

So the apparent ~5–6 kJ/mol "second basin" there is an artifact of **incomplete walker mixing**,
not a resolved state, and the ergodicity mask (`min_walkers = 3`) removes it. S26D's global
minimum (bend ≈ 19°, $n_\mathrm{IDR-DNA}$ ≈ 11) and every other system's basins are unaffected.
Fully resolving that corner would need better walker exchange — a new simulation, out of scope
for this figure.

### Panel S2 — DNA-bending PMF, WT vs mutants vs DNA alone

1D reweighted $F(\mathrm{bend})$ overlay, the mutant extension of **Panel D1**: how each
point mutation shifts the preferred DNA bend relative to WT holo and to the free DNA
reference. Same reliability truncation (`STD_CEIL = 15` kJ/mol) and axes as Panel D1.
Colours follow the Okabe–Ito / Wong palette used in the wall-system comparison figures.

**Uncertainty caveat.** The bend angle is an *unbiased spectator* (only `coord_tail_dna` and
`d_e2e` are biased), and the six walkers are still segregated in bend at the end of every run —
the spread of per-walker median bend is ≈ 23° (WT), 29° (TM), 34° (S26D), 38° (T63D), i.e. as
wide as the whole inter-system effect being measured. Extending WT/S26D/T63D from 300 to 350 ns
did not close this: the spreads are essentially the 300 ns ones. The bend ordering below
(DNA 8° < TM 20° < S26D 22° < WT 31° < T63D 37°) is what the reweighted PMFs give and is
unchanged by the extension — only T63D's minimum moved at all (38.2° → 36.8°), and it stays the
most bent. But because the mutant curves sit on less-mixed landscapes than the WT/DNA
references, the plotted time-snapshot ±std bands likely *under*state the true (cross-walker)
uncertainty on them. Treat the ordering as robust; treat the exact mutant minima as ± a few
degrees. This is a walker-mixing limit, not a run-length limit, so TM's shorter 300 ns is not
what dominates its error bar.

In [ ]:
# ─── Panel S2: 1D F(bend), DNA alone vs WT vs the three mutants ───────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — S2 (matched to Panel D1)
VERSION  = "v2"             # v2: WT/S26D/T63D tables rebuilt at 350 ns (TM still 300 ns)
FIGSIZE  = (4.6, 3.4)
XLIM     = (0, 60)          # degrees
YLIM     = (-1.5, 20)          # kJ/mol
STD_CEIL = 15.0
OUT      = "mutants_pmf_bend_vs_wt_dna"
# system stem in analysis/<sys>_pmf_bend_mean_std.dat -> (label, colour, linestyle).
# Legend labels carry the run length: it is NOT uniform (see the header cell).
SYS = [("DNA_wall2",   "DNA", "#0072B2", "-"),   # Okabe-Ito blue
       ("1Nhp6A_wall", "WT (Nhp6A)", "#D55E00", "-"),   # vermillion (WT, established)
       ("1S26D_wall",  "S26D",       "#CC79A7", "-"),   # reddish purple
       ("1T63D_wall",  "T63D",       "#E69F00", "-"),   # orange
       ("1TM_wall",    "TM",         "#56B4E9", "-")]   # sky blue

plt.rcParams.update({"font.size": 12, "axes.labelsize": 11, "xtick.labelsize": 10,
                     "ytick.labelsize": 10, "legend.fontsize": 8.5, "axes.linewidth": 0.8,
                     "figure.dpi": 150, "axes.grid": False, "pdf.fonttype": 42})

fig, ax = plt.subplots(figsize=FIGSIZE)
for name, label, color, ls in SYS:
    d = np.loadtxt(f"analysis/{name}_pmf_bend_mean_std.dat")
    x, m, sd = d[:, 0], d[:, 1], d[:, 2]
    ok = f5.reliable_1d(m, sd, STD_CEIL)
    ax.plot(x[ok], m[ok], lw=2, ls=ls, color=color, label=label)
    ax.fill_between(x[ok], (m - sd)[ok], (m + sd)[ok], color=color, alpha=0.18, lw=0)
    print(f"  {label:20s} minimum at {x[ok][np.nanargmin(m[ok])]:5.1f} deg"
          f"   (reliable {x[ok].min():.1f} – {x[ok].max():.1f} deg)")
ax.set_xlabel("DNA bend angle (deg)")
ax.set_ylabel(r"$\Delta F$ (kJ mol$^{-1}$)")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.set_yticks(range(0, 21, 5))
ax.legend(frameon=False, loc="upper right", ncol=2)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE, transparent=False)
plt.show()

### Panel S3 — IDR–DNA contact-number PMF, WT vs mutants

1D reweighted $F(n_\mathrm{IDR-DNA})$ overlay, the contact-number companion to Panel S2's
bend-angle overlay: how each point mutation shifts the preferred number of IDR–DNA contacts
relative to WT holo. Same reweighting (5-snapshot final-bias ensemble), reliability
truncation (`STD_CEIL = 15` kJ/mol), and Okabe–Ito/Wong colour code as Panel S2 and the
main wall-system comparison figures. No free-DNA reference here — `n_IDR-DNA` is only
defined for the protein:DNA systems (`coord_tail_dna` is one of their biased CVs; free DNA
has no protein and no such CV).

In [ ]:
# ─── Panel S3: 1D F(n_IDR-DNA), WT vs the three mutants ───────────────────────────
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import numpy as np
import matplotlib.pyplot as plt
import article_plot as f5

# settings — S3 (matched to Panel S2)
VERSION  = "v2"             # v2: WT/S26D/T63D tables rebuilt at 350 ns (TM still 300 ns)
FIGSIZE  = (4.6, 3.4)
XLIM     = (0, 35)          # n_IDR-DNA contacts
YLIM     = (-1.5, 20)       # kJ/mol
STD_CEIL = 15.0
OUT      = "mutants_pmf_nidr_vs_wt"
# system stem in analysis/<sys>_pmf_nidr_mean_std.dat -> (label, colour, linestyle).
# Legend labels carry the run length: it is NOT uniform (see the header cell).
SYS = [("1Nhp6A_wall", "WT (Nhp6A), 350 ns", "#D55E00", "-"),   # vermillion (WT, established)
       ("1S26D_wall",  "S26D, 350 ns",       "#CC79A7", "-"),   # reddish purple
       ("1T63D_wall",  "T63D, 350 ns",       "#E69F00", "-"),   # orange
       ("1TM_wall",    "TM, 300 ns",         "#56B4E9", "-")]   # sky blue

plt.rcParams.update({"font.size": 12, "axes.labelsize": 11, "xtick.labelsize": 10,
                     "ytick.labelsize": 10, "legend.fontsize": 8.5, "axes.linewidth": 0.8,
                     "figure.dpi": 150, "axes.grid": False, "pdf.fonttype": 42})

fig, ax = plt.subplots(figsize=FIGSIZE)
for name, label, color, ls in SYS:
    d = np.loadtxt(f"analysis/{name}_pmf_nidr_mean_std.dat")
    x, m, sd = d[:, 0], d[:, 1], d[:, 2]
    ok = f5.reliable_1d(m, sd, STD_CEIL)
    ax.plot(x[ok], m[ok], lw=2, ls=ls, color=color, label=label)
    ax.fill_between(x[ok], (m - sd)[ok], (m + sd)[ok], color=color, alpha=0.18, lw=0)
    print(f"  {label:20s} minimum at {x[ok][np.nanargmin(m[ok])]:5.1f} contacts"
          f"   (reliable {x[ok].min():.1f} – {x[ok].max():.1f})")
ax.set_xlabel(r"$n_\mathrm{IDR-DNA}$ contacts")
ax.set_ylabel(r"$\Delta F$ (kJ mol$^{-1}$)")
ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.legend(frameon=False, loc="upper right", ncol=2)
f5.savepanel(fig, OUT, VERSION, figs=f5.ARTICLE, transparent=False)
plt.show()

### Provenance stamp

Writes `figures/article/article_figures_provenance.txt` — which dataset produced each panel, how long it ran, which
gates it passed, what is still preliminary. Keep it with the figure: it is what makes the panels
traceable when the paper comes back from review.

In [ ]:
# ─── Provenance stamp: everything this cell needs ─────────────────────────────
import datetime
import socket
import os, sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis/scripts").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "analysis/scripts"))   # project root, wherever this notebook lives
import article_plot as f5

VERSION = "v1"          # keep in step with the panels above
STD_CEIL, MIN_VISITS, ESS_MIN_BIN = 10.0, 5, 30    # the filters the panels applied

META = f5.load_meta()
lines = [f"Article figures — provenance ({VERSION})",
         f"panels rendered {datetime.datetime.now():%Y-%m-%d %H:%M} on {socket.gethostname()} "
         f"by notebooks/article/article_figures.ipynb",
         f"data computed {META['generated']} on {META['host']} by analysis/scripts/article_data.py "
         f"(stride {META['stride']}, {META['n_snaps']} snapshots)",
         f"plot-time filters: std ceiling {STD_CEIL} kJ/mol, min visits {MIN_VISITS}, "
         f"ESS floor {ESS_MIN_BIN}",
         ""]
for k in ("A", "B", "C", "D", "E", "F1", "F2", "reweighting_apo"):
    if k in META:
        lines.append(f"[{k}]")
        for kk, vv in META[k].items():
            lines.append(f"    {kk:20s} {vv}")
        lines.append("")
lines += ["FLAGS"] + [f"  - {f}" for f in META["flags"]]

os.makedirs(f5.ARTICLE, exist_ok=True)
with open("figures/article/article_figures_provenance.txt", "w") as fh:
    fh.write("\n".join(lines) + "\n")
print("\n".join(lines))
print("\nwrote figures/article/article_figures_provenance.txt")